# Notebook 5: Full RAG Pipeline Execution

**Goal**: Run the complete end-to-end RAG pipeline including advanced modes.

## Topics:
- Basic RAG query
- Multi-turn conversation (session memory)
- Multi-hop retrieval
- Agentic RAG (ReAct)
- Streaming responses
- Performance profiling


In [ ]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')
import os

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

from config import RAGConfig, LLMConfig, EmbeddingConfig, ChunkingConfig, VectorStoreConfig, RetrievalConfig
from rag_pipeline import RAGPipeline
from embeddings.embedding import EmbeddingEngine
from vector_store.vector_store import VectorStoreFactory
from generation.llm_interface import LLMInterface
from retrieval.reranker import CrossEncoderReranker

print('RAG System modules loaded')
print(f'OpenAI API key set: {bool(os.getenv("OPENAI_API_KEY"))}')

## 5.1 Build Pipeline

Choose between OpenAI (requires API key) or Ollama (local, free).

In [ ]:
# Option A: OpenAI (requires OPENAI_API_KEY)
USE_OPENAI = bool(os.getenv('OPENAI_API_KEY'))

# Embedding engine (local, no API key needed)
embedder = EmbeddingEngine(
    provider='sentence_transformers',
    model_name='all-MiniLM-L6-v2',
    use_cache=True,
    cache_dir='../.embed_cache',
    device='cpu',
)

# Vector store
vector_store = VectorStoreFactory.create(
    backend='faiss',
    dimension=embedder.dimension,
    index_type='flat',
    metric='cosine',
)

# LLM — use OpenAI if key available, else Ollama
if USE_OPENAI:
    llm = LLMInterface(provider='openai', model='gpt-4o-mini', temperature=0.0)
    print('Using OpenAI gpt-4o-mini')
else:
    llm = LLMInterface(provider='ollama', model='llama3.1:8b', temperature=0.0)
    print('Using Ollama llama3.1:8b (make sure `ollama serve` is running)')

# Reranker
reranker = CrossEncoderReranker(model_name='cross-encoder/ms-marco-MiniLM-L-6-v2')

# Build pipeline
pipeline = RAGPipeline(
    embedder=embedder,
    vector_store=vector_store,
    llm=llm,
    reranker=reranker,
)
print('Pipeline initialized!')

## 5.2 Index Documents

In [ ]:
result = pipeline.index(['../sample_data/'])

print(f'Indexing complete:')
print(f'  Documents:     {result.num_documents}')
print(f'  Chunks:        {result.num_chunks}')
print(f'  Embed latency: {result.embedding_latency_ms:.0f} ms')
print(f'  Total latency: {result.total_latency_ms:.0f} ms')
print(f'  Index size:    {pipeline.index_size} vectors')
if result.errors:
    print(f'  Errors:        {result.errors}')

## 5.3 Basic RAG Query

In [ ]:
def show_rag_response(response):
    print(f'Q: {response.query}')
    print(f'\nA: {response.answer}')
    print(f'\nCitations:')
    for c in response.citations:
        print(f'  [{c["ref"]}] {c.get("source","?").split(chr(92))[-1].split("/")[-1]} (score={c["score"]})')
    print(f'\nLatency breakdown:')
    for k, v in response.latency_breakdown.items():
        print(f'  {k}: {v:.1f} ms')
    print(f'  TOTAL: {response.total_latency_ms:.1f} ms')

response = pipeline.query('What is the refund policy?')
show_rag_response(response)

In [ ]:
response2 = pipeline.query('What payment methods do you accept?')
show_rag_response(response2)

In [ ]:
# Question outside the knowledge base
response3 = pipeline.query('What is the capital of France?')
show_rag_response(response3)

## 5.4 Multi-Turn Conversation (Session Memory)

In [ ]:
SESSION_ID = 'demo-session-001'

conv_queries = [
    'How do I return a product?',
    'How long does the refund take?',
    'What if it was a final sale item?',    # refers to previous context
]

for i, q in enumerate(conv_queries, 1):
    print(f'Turn {i}: {q}')
    r = pipeline.query(q, session_id=SESSION_ID)
    print(f'Answer: {r.answer[:200]}...')
    print('-' * 50)

## 5.5 Template Variants

In [ ]:
q = 'What are the product return options?'

for template in ['default', 'chain_of_thought', 'with_citations']:
    print(f'\n=== Template: {template} ===')
    r = pipeline.query(q, template=template)
    print(r.answer[:300])
    print('...')

## 5.6 Metadata Filtering

In [ ]:
# Only search within FAQ document
r_filtered = pipeline.query(
    'How do I track my order?',
    filter={'file_type': 'md'},
)

print('Filtered to Markdown (FAQ) only:')
print(f'Answer: {r_filtered.answer}')
print(f'\nSources used: {[c.get("source","?").split(chr(92))[-1] for c in r_filtered.citations]}')

## 5.7 Multi-Hop Query Demo

In [ ]:
# Multi-hop: requires combining info from multiple chunks
multi_hop_q = 'If I bought a SmartWatch and want to return it after 35 days, what are my options?'

print(f'Multi-hop query: {multi_hop_q}')
print('(This requires combining product info + return policy info)\n')

mh_response = pipeline.multi_hop_query(multi_hop_q, max_hops=3)

print(f'Answer: {mh_response.answer}')
print(f'\nChunks retrieved across hops: {len(mh_response.retrieved_chunks)}')
print(f'Metadata: {mh_response.metadata}')

## 5.8 Streaming Response

In [ ]:
print('Streaming response (token by token):')
print('-' * 50)

for token in pipeline.stream_query('Describe all products available'):
    print(token, end='', flush=True)

print('\n' + '-' * 50)
print('Stream complete')

## 5.9 Performance Profile

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

test_questions = [
    'What is the refund policy?',
    'How long does shipping take?',
    'What payment methods are accepted?',
    'Tell me about the laptop product',
    'How do I track my package?',
]

latency_data = {'retrieval': [], 'generation': [], 'total': []}

for q in test_questions:
    r = pipeline.query(q)
    latency_data['retrieval'].append(r.latency_breakdown.get('retrieval_ms', 0))
    latency_data['generation'].append(r.latency_breakdown.get('generation_ms', 0))
    latency_data['total'].append(r.total_latency_ms)

df_lat = pd.DataFrame(latency_data, index=[q[:30] for q in test_questions])
print('Latency breakdown per query (ms):')
print(df_lat.to_string())
print(f'\nMean total: {np.mean(latency_data["total"]):.0f}ms')
print(f'P90 total:  {np.percentile(latency_data["total"], 90):.0f}ms')

In [ ]:
import pandas as pd

fig, ax = plt.subplots(figsize=(10, 5))
df_lat[['retrieval', 'generation']].plot(
    kind='bar', stacked=True, ax=ax,
    color=['#2196F3', '#FF9800'],
)
ax.set_title('RAG Latency Breakdown by Query')
ax.set_xlabel('Query')
ax.set_ylabel('Latency (ms)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('rag_latency.png', dpi=100)
plt.show()

## 5.10 Save Pipeline State

In [ ]:
pipeline.save('../pipeline_state')
print('Pipeline saved to ../pipeline_state')

# Later, reload:
# pipeline.load('../pipeline_state')
# print(f'Reloaded. Index size: {pipeline.index_size}')

## RAG vs Fine-Tuning Decision Matrix

| Criteria | RAG | Fine-Tuning | RAG + Fine-Tuning |
|---|---|---|---|
| **Knowledge freshness** | ✅ Real-time | ❌ Frozen at train time | ✅ Real-time |
| **Knowledge update cost** | ✅ Re-index only | ❌ Full retraining | ✅ Re-index only |
| **Hallucination control** | ✅ Grounded in docs | ⚠️ Can hallucinate | ✅ Best |
| **Response style/format** | ⚠️ Prompt engineering | ✅ Native via examples | ✅ Best |
| **Specialized vocabulary** | ⚠️ If in chunks | ✅ Learned in weights | ✅ Best |
| **Latency** | ⚠️ +retrieve overhead | ✅ Single forward pass | ⚠️ +retrieve overhead |
| **Cost (inference)** | ⚠️ Larger context | ✅ Shorter prompts | ⚠️ Larger context |
| **Privacy (data exposure)** | ⚠️ Chunks in prompt | ✅ Weights only | ⚠️ Chunks in prompt |
| **Auditability** | ✅ Citations | ❌ Black box | ✅ Citations |

**When to combine**: Use a fine-tuned model (domain language, output format) + RAG (current facts, private data).
Example: Fine-tune on medical terminology → RAG retrieves the latest clinical guidelines.